In [3]:
pip install numpy scikit-learn xgboost lightgbm


  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl.metadata (17 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (8.9 MB)
Using cached xgboost-3.2.0-py3-none-manylinux_2_28_x86_64.whl (131.7 MB)
Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl (3.6 MB)
Using cached joblib-1.5.3-py3-none-any.whl

In [4]:
import numpy as np
from sklearn.datasets import fetch_covtype
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier
import time
import warnings
warnings.filterwarnings('ignore')

print("=== SVM-based Data Reduction Pipeline (Final Fixed Version) ===\n")

# ========================== 1. Load Covertype Dataset ==========================
print("Loading Covertype Dataset...")
data = fetch_covtype()
X = data.data
y = data.target

# Convert labels from 1-7 to 0-6
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print(f"Original Dataset Shape: {X.shape} | Classes: {len(np.unique(y))}\n")

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

# Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ========================== 2. Train Initial SVM ==========================
print("Training Initial SVM using LinearSVC...")
start_time = time.time()

svm = LinearSVC(C=5.0, dual=False, random_state=42, max_iter=10000, tol=1e-3)
svm.fit(X_train, y_train)

print(f"Initial SVM Training Completed! Time: {time.time() - start_time:.1f} seconds\n")

# ========================== 3. Extract Approximate Support Vectors ==========================
print("Extracting points near decision boundary...")

decision_scores = svm.decision_function(X_train)
max_scores = np.max(np.abs(decision_scores), axis=1)
margin_mask = max_scores <= 1.2

X_sv = X_train[margin_mask]
y_sv = y_train[margin_mask]

print(f"Approximate Support Vectors: {X_sv.shape[0]:,} samples "
      f"({X_sv.shape[0]/X_train.shape[0]*100:.2f}% of train data)")

# ========================== 4. SV Augmentation ==========================
np.random.seed(42)
noise = np.random.normal(0, 0.05, X_sv.shape)
X_aug_sv = X_sv + noise
y_aug_sv = y_sv.copy()

X_sv_aug = np.vstack([X_sv, X_aug_sv])
y_sv_aug = np.concatenate([y_sv, y_aug_sv])

print(f"After SV Augmentation: {X_sv_aug.shape[0]:,} samples")

# ========================== 5. Sensitivity-Guided Sampling ==========================
non_sv_mask = ~margin_mask
X_non_sv = X_train[non_sv_mask]
y_non_sv = y_train[non_sv_mask]

dist_from_margin = np.max(np.abs(svm.decision_function(X_non_sv)), axis=1)

n_sample = int(0.12 * len(X_non_sv))
idx = np.argsort(dist_from_margin)[-n_sample:]

X_sampled = X_non_sv[idx]
y_sampled = y_non_sv[idx]

print(f"Sensitivity-Guided Sampling: {X_sampled.shape[0]:,} samples")

# ========================== 6. Merge ==========================
X_merged = np.vstack([X_sv_aug, X_sampled])
y_merged = np.concatenate([y_sv_aug, y_sampled])

# CRITICAL FIX: Re-encode labels after sampling (to make them continuous from 0)
final_label_encoder = LabelEncoder()
y_merged = final_label_encoder.fit_transform(y_merged)

print(f"\nFinal Reduced Dataset Shape: {X_merged.shape}")
print(f"Reduction from original train set: {100 - (X_merged.shape[0]/X_train.shape[0]*100):.2f}%\n")

# ========================== 7. Train Final Ensemble ==========================
print("Training Final Ensemble Models...")

models = {
    'XGBoost': xgb.XGBClassifier(n_estimators=150, max_depth=7, learning_rate=0.1, 
                                 random_state=42, eval_metric='mlogloss'),
    'LightGBM': lgb.LGBMClassifier(n_estimators=150, max_depth=7, learning_rate=0.1, 
                                   random_state=42, verbose=-1),
    'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=200, random_state=42)
}

for name, model in models.items():
    print(f"  Training {name}...")
    model.fit(X_merged, y_merged)

# Final Voting Ensemble
ensemble = VotingClassifier(
    estimators=[('xgb', models['XGBoost']), 
                ('lgb', models['LightGBM']), 
                ('mlp', models['MLP'])],
    voting='soft'
)
ensemble.fit(X_merged, y_merged)

# ========================== 8. Evaluation ==========================
y_pred = ensemble.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("\n" + "="*60)
print("FINAL RESULTS")
print("="*60)
print(f"Original Train Samples     : {X_train.shape[0]:,}")
print(f"Reduced Dataset (X_reduced): {X_merged.shape[0]:,} samples")
print(f"Data Reduction Achieved    : {100 - (X_merged.shape[0]/X_train.shape[0]*100):.2f}%")
print(f"Final Ensemble Accuracy    : {acc*100:.3f}%")
print("="*60)

# Save reduced dataset
np.savez("reduced_covertype.npz", X_reduced=X_merged, y_reduced=y_merged)
print("\nReduced dataset saved as 'reduced_covertype.npz'")

=== SVM-based Data Reduction Pipeline (Final Fixed Version) ===

Loading Covertype Dataset...
Original Dataset Shape: (581012, 54) | Classes: 7

Training Initial SVM using LinearSVC...
Initial SVM Training Completed! Time: 66.7 seconds

Extracting points near decision boundary...
Approximate Support Vectors: 31 samples (0.01% of train data)
After SV Augmentation: 62 samples
Sensitivity-Guided Sampling: 55,773 samples

Final Reduced Dataset Shape: (55835, 54)
Reduction from original train set: 87.99%

Training Final Ensemble Models...
  Training XGBoost...
  Training LightGBM...
  Training MLP...

FINAL RESULTS
Original Train Samples     : 464,809
Reduced Dataset (X_reduced): 55,835 samples
Data Reduction Achieved    : 87.99%
Final Ensemble Accuracy    : 63.069%

Reduced dataset saved as 'reduced_covertype.npz'
